## Finguard Secret Vault - Secret Scope Setup

This notebook creates a Databricks secret scope named **`finguard secret vault`** to securely store credentials for the FinGuard streaming project.

**Secrets stored:**
| Key | Purpose |
|-----|--------|
| KAFKA_BOOTSTRAP_SERVERS | Confluent Cloud Kafka broker endpoint |
| KAFKA_TOPIC | Target Kafka topic for credit card transactions |
| API_KEY | Confluent Cloud API key |
| API_SECRET | Confluent Cloud API secret |

> **Note:** Run this notebook once to provision the scope. Future notebooks can access secrets via `dbutils.secrets.get("finguard secret vault", "<key>")`

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

SCOPE_NAME = "finguard secret vault"

# Create the secret scope
try:
    w.secrets.create_scope(scope=SCOPE_NAME)
    print(f"✓ Secret scope '{SCOPE_NAME}' created successfully")
except Exception as e:
    if "RESOURCE_ALREADY_EXISTS" in str(e):
        print(f"ℹ Secret scope '{SCOPE_NAME}' already exists - proceeding to add secrets")
    else:
        raise e

# Define the Kafka connection secrets
secrets = {
    "KAFKA_BOOTSTRAP_SERVERS": "pkc-n3603.us-central1.gcp.confluent.cloud:9092",
    "KAFKA_TOPIC": "credit_card_transactions",
    "API_KEY": "NASKQKANPTF36VS4",
    "API_SECRET": "cfltOJJ5C/i0pVbNZAmxvoEvrYKUwoyP1xcbUUudj+tOxGXpkcJIVB3W5TUvYb2Q"
}

# Store each secret in the scope
for key, value in secrets.items():
    w.secrets.put_secret(scope=SCOPE_NAME, key=key, string_value=value)
    print(f"✓ Secret added: {key}")

print(f"\n✅ All secrets stored in scope '{SCOPE_NAME}'")

In [0]:
# Verify the scope and secrets were created
print("=" * 50)
print(f"Scope: {SCOPE_NAME}")
print("=" * 50)

# List all secrets in the scope
secret_list = w.secrets.list_secrets(scope=SCOPE_NAME)
for secret in secret_list:
    print(f"  🔑 {secret.key}")

print("\n--- Verification (values are redacted by Databricks) ---")
for key in secrets.keys():
    val = dbutils.secrets.get(scope=SCOPE_NAME, key=key)
    print(f"  {key}: {val}")

In [0]:
%sql select * from finguard.silver.transactions where silver_TS is not null;

drop table finguard.silver.transactions;
